In [1]:
#fill NA with None
from sklearn import tree
from sklearn.preprocessing import LabelEncoder
import pandas as pd
import random
import numpy as np
from sklearn.metrics import accuracy_score
from sklearn.model_selection import KFold
from statistics import mean
from sklearn.naive_bayes import BernoulliNB

In [2]:
#https://stackoverflow.com/questions/20224526/how-to-extract-the-decision-rules-from-scikit-learn-decision-tree
def get_code(tree, feature_names):
        left      = tree.tree_.children_left
        right     = tree.tree_.children_right
        threshold = tree.tree_.threshold
        features  = [feature_names[i] for i in tree.tree_.feature]
        value = tree.tree_.value

        def recurse(left, right, threshold, features, node):
                if (threshold[node] != -2):
                        print("if ( " + features[node] + " <= " + str(threshold[node]) + " ) {")
                        if left[node] != -1:
                                recurse (left, right, threshold, features,left[node])
                        print( "} else {")
                        if right[node] != -1:
                                recurse (left, right, threshold, features,right[node])
                        print( "}")
                else:
                        print( "return " + str(value[node]))

        recurse(left, right, threshold, features, 0)

In [3]:
data = pd.read_excel('dataset_gen.xlsx') 
dataset_all_gen = pd.read_excel('dataset_all_gen.xlsx')
dataset_all_original = pd.read_excel('dataset_all_original.xlsx')
dataset_jaccard_gen = pd.read_excel('dataset_jaccard_gen.xlsx')
dataset_jaccard_original = pd.read_excel('dataset_jaccard_original.xlsx')
dataset_myroom_gen = pd.read_excel('dataset_myroom_gen.xlsx')
dataset_myroom_original = pd.read_excel('dataset_myroom_original.xlsx')
dataset_university_gen = pd.read_excel('dataset_university_gen.xlsx')
dataset_university_original = pd.read_excel('dataset_university_original.xlsx')

In [4]:
data = data.sample(frac = 1) 
dataset_all_gen = dataset_all_gen.sample(frac = 1) 
dataset_all_original = dataset_all_original.sample(frac = 1) 
dataset_jaccard_gen = dataset_jaccard_gen.sample(frac = 1) 
dataset_jaccard_original = dataset_jaccard_original.sample(frac = 1) 
dataset_myroom_gen = dataset_myroom_gen.sample(frac = 1) 
dataset_myroom_original = dataset_myroom_original.sample(frac = 1) 
dataset_university_gen = dataset_university_gen.sample(frac = 1) 
dataset_university_original = dataset_university_original.sample(frac = 1) 

In [5]:

data = data.reset_index(drop=True)
dataset_all_gen = dataset_all_gen.reset_index(drop=True)
dataset_all_original = dataset_all_original.reset_index(drop=True)
dataset_jaccard_gen = dataset_jaccard_gen.reset_index(drop=True)
dataset_jaccard_original = dataset_jaccard_original.reset_index(drop=True)
dataset_myroom_gen = dataset_myroom_gen.reset_index(drop=True)
dataset_myroom_original = dataset_myroom_original.reset_index(drop=True)
dataset_university_gen = dataset_university_gen.reset_index(drop=True)
dataset_university_original = dataset_university_original.reset_index(drop=True)

In [6]:
data.isnull().sum()

type             0
Name             0
Vendor           0
DNS           2304
mDNS          2704
NUPNP        11000
open_port     1731
SSDP_Type     2304
SSDP_t       11000
SSDP_v       11000
dtype: int64

In [7]:
data = data.fillna("NaN")

In [8]:
#fill Na
dataset_all_gen = dataset_all_gen.fillna("NaN")
dataset_all_original = dataset_all_original.fillna("NaN")
dataset_jaccard_gen = dataset_jaccard_gen.fillna("NaN")
dataset_jaccard_original = dataset_jaccard_original.fillna("NaN")
dataset_myroom_gen = dataset_myroom_gen.fillna("NaN")
dataset_myroom_original = dataset_myroom_original.fillna("NaN")
dataset_university_gen = dataset_university_gen.fillna("NaN")
dataset_university_original = dataset_university_original.fillna("NaN")

In [9]:
data.isnull().sum()

type         0
Name         0
Vendor       0
DNS          0
mDNS         0
NUPNP        0
open_port    0
SSDP_Type    0
SSDP_t       0
SSDP_v       0
dtype: int64

In [10]:
data.dtypes

type         object
Name         object
Vendor       object
DNS          object
mDNS         object
NUPNP        object
open_port    object
SSDP_Type    object
SSDP_t       object
SSDP_v       object
dtype: object

In [11]:
data = data.astype('category')

In [12]:
dataset_all_gen = dataset_all_gen.astype('category')
dataset_all_original = dataset_all_original.astype('category')
dataset_jaccard_gen = dataset_jaccard_gen.astype('category')
dataset_jaccard_original = dataset_jaccard_original.astype('category')
dataset_myroom_gen = dataset_myroom_gen.astype('category')
dataset_myroom_original = dataset_myroom_original.astype('category')
dataset_university_gen = dataset_university_gen.astype('category')
dataset_university_original = dataset_university_original.astype('category')

In [13]:
data.dtypes

type         category
Name         category
Vendor       category
DNS          category
mDNS         category
NUPNP        category
open_port    category
SSDP_Type    category
SSDP_t       category
SSDP_v       category
dtype: object

In [14]:
data

,type,Name,Vendor,DNS,mDNS,NUPNP,open_port,SSDP_Type,SSDP_t,SSDP_v
0,Cameras,Dropcam,Dropcam,telegraph.co.uk,NaN,NaN,tcp/443,NOTIFY * HTTP/1.1,NaN,NaN
1,Cameras,"Insteon Camera, wifi",Insteon,wufoo.com,_zoolab._tcp.local.,NaN,tcp/1883,NOTIFY * HTTP/1.1,NaN,NaN
2,Appliances,iHome,SDI Technologies Inc,craigslist.org,NaN,NaN,NaN,NOTIFY * HTTP/1.1,NaN,NaN
3,Non-IOT,Samsung s7,SAMSUNG,NaN,_bigtax._tcp.local.,NaN,tcp/443,NaN,NaN,NaN
4,Non-IOT,IPhone,"Apple, Inc.",miitbeian.gov.cn,_matsoft._tcp.local.,NaN,tcp/22,M-SEARCH * HTTP/1.1,NaN,NaN
5,Non-IOT,Nexus 5x Smartphone,LG Electronics,twitpic.com,_mat lam tam._tcp.local.,NaN,tcp/1883,M-SEARCH * HTTP/1.1,NaN,NaN
6,Non-IOT,Dell Laptop win 10,Dell,mozilla.com,_quo lux._tcp.local.,NaN,tcp/1883,M-SEARCH * HTTP/1.1,NaN,NaN
7,Energy Management,Belkin WeMo Link,Belkin International Inc.,NaN,NaN,NaN,tcp/22,NaN,NaN,NaN
8,Energy Management,TP-Link Smart WiFi LED Bulb,"TP-LINK TECHNOLOGIES CO.,LTD.",booking.com,NaN,NaN,tcp/1900,NOTIFY * HTTP/1.1,NaN,NaN
9,Hubs,Wink 2 Hub,WINK LABS INC,naver.com,NaN,NaN,tcp/22,NOTIFY * HTTP/1.1,NaN,NaN


In [15]:
y = data.drop_duplicates(keep='first')
y = y.drop(['Name'], axis=1)
y = y.drop_duplicates(keep='first')
# y = y.reset_index()
# shuffle the DataFrame rows
# y = y.sample(frac = 1) 


In [16]:
len(y)

10610

In [17]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original.drop_duplicates(keep='first')

In [18]:
#drop name
dataset_all_gen_y = dataset_all_gen_y.drop(['Name'], axis=1)
dataset_all_original_y = dataset_all_original_y.drop(['Name'], axis=1)
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop(['Name'], axis=1)
dataset_jaccard_original_y = dataset_jaccard_original_y.drop(['Name'], axis=1)
dataset_myroom_gen_y = dataset_myroom_gen_y.drop(['Name'], axis=1)
dataset_myroom_original_y = dataset_myroom_original_y.drop(['Name'], axis=1)
dataset_university_gen_y = dataset_university_gen_y.drop(['Name'], axis=1)
dataset_university_original_y = dataset_university_original_y.drop(['Name'], axis=1)

In [19]:
#drop_duplicates
dataset_all_gen_y = dataset_all_gen_y.drop_duplicates(keep='first')
dataset_all_original_y = dataset_all_original_y.drop_duplicates(keep='first')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.drop_duplicates(keep='first')
dataset_jaccard_original_y = dataset_jaccard_original_y.drop_duplicates(keep='first')
dataset_myroom_gen_y = dataset_myroom_gen_y.drop_duplicates(keep='first')
dataset_myroom_original_y = dataset_myroom_original_y.drop_duplicates(keep='first')
dataset_university_gen_y = dataset_university_gen_y.drop_duplicates(keep='first')
dataset_university_original_y = dataset_university_original_y.drop_duplicates(keep='first')

In [20]:
y = y.apply(LabelEncoder().fit_transform)

In [21]:
dataset_all_gen_y = dataset_all_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_all_original_y = dataset_all_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_gen_y = dataset_jaccard_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_jaccard_original_y = dataset_jaccard_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_gen_y = dataset_myroom_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_myroom_original_y = dataset_myroom_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_gen_y = dataset_university_gen_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')
dataset_university_original_y = dataset_university_original_y.apply(lambda col: LabelEncoder().fit_transform(col.astype(str)), axis=0, result_type='expand')

In [22]:
list_data = []
list_data.append(y)
list_data.append(dataset_all_gen_y)
list_data.append(dataset_all_original_y)
list_data.append(dataset_jaccard_gen_y)
list_data.append(dataset_jaccard_original_y)
list_data.append(dataset_myroom_gen_y)
list_data.append(dataset_myroom_original_y)
list_data.append(dataset_university_gen_y)
list_data.append(dataset_university_original_y)
len(list_data)

9

In [23]:
accuracy = []
accuracy_t = []
accuracy_temp = []

In [24]:
for data in list_data:
    for i in range(100):
        kf = KFold(n_splits=10, random_state=None, shuffle=True)
#         kf.get_n_splits(data)
        for train_index, test_index in kf.split(data):
        #     print(index)
#             print("TRAIN:", train_index, "TEST:", test_index)
            train, test = data.iloc[train_index], data.iloc[test_index]
#             print("TRAIN:", train, "TEST:", test)
        #     print(len(train))
#             print("TRAIN:", len(train), "TEST:", len(test))
            label_data = train.type
            train_data = train.drop(['type'], axis=1)
            tlabel_data = test.type
            test_data = test.drop(['type'], axis=1)
            truelabel_data = pd.DataFrame(tlabel_data)
            truelabel_data = truelabel_data.reset_index(drop=True)
            clf = BernoulliNB()
#             clf = tree.DecisionTreeClassifier()
#             clf = clf.fit(train_data,label_data )
            clf = clf.fit(train_data,label_data )
#             predict = clf.predict(test_data)
            predict = clf.predict(test_data)
#             print(truelabel_data)
#             print(predict)
        #     len(predict)
            result_label_data = pd.DataFrame(columns = ['type'])
            result_label_data.type = predict
            compare = pd.DataFrame(columns = ['truelabel_data','predictedlabel'])
            compare.truelabel_data = truelabel_data.type
            compare.predictedlabel = result_label_data.type
#             print(compare)
            t = (accuracy_score(compare.truelabel_data, compare.predictedlabel))
            accuracy_temp.append(t)
            print(i,"  Accuracy: ",t)
        accuracy_t.append(mean(accuracy_temp))
    accuracy.append(mean(accuracy_t))

0   Accuracy:  0.49104618284637136
0   Accuracy:  0.49104618284637136
0   Accuracy:  0.473138548539114
0   Accuracy:  0.4816211121583412
0   Accuracy:  0.47785108388312914
0   Accuracy:  0.4637134778510839
0   Accuracy:  0.49198868991517436
0   Accuracy:  0.49764373232799247
0   Accuracy:  0.4816211121583412
0   Accuracy:  0.46936852026390197
1   Accuracy:  0.49198868991517436
1   Accuracy:  0.46182846371347785
1   Accuracy:  0.49010367577756836
1   Accuracy:  0.5014137606032045
1   Accuracy:  0.48350612629594725
1   Accuracy:  0.45617342130065974
1   Accuracy:  0.49764373232799247
1   Accuracy:  0.48350612629594725
1   Accuracy:  0.49104618284637136
1   Accuracy:  0.46182846371347785
2   Accuracy:  0.4806786050895382
2   Accuracy:  0.49293119698397736
2   Accuracy:  0.4505183788878417
2   Accuracy:  0.472196041470311
2   Accuracy:  0.46088595664467485
2   Accuracy:  0.4759660697455231
2   Accuracy:  0.4825636192271442
2   Accuracy:  0.4948162111215834
2   Accuracy:  0.4910461828463713

23   Accuracy:  0.5004712535344015
23   Accuracy:  0.49764373232799247
23   Accuracy:  0.45428840716305374
23   Accuracy:  0.4740810556079171
24   Accuracy:  0.4957587181903864
24   Accuracy:  0.46936852026390197
24   Accuracy:  0.4872761545711593
24   Accuracy:  0.4957587181903864
24   Accuracy:  0.4750235626767201
24   Accuracy:  0.4740810556079171
24   Accuracy:  0.4825636192271442
24   Accuracy:  0.4646559849198869
24   Accuracy:  0.5014137606032045
24   Accuracy:  0.473138548539114
25   Accuracy:  0.5127238454288408
25   Accuracy:  0.47785108388312914
25   Accuracy:  0.4750235626767201
25   Accuracy:  0.46842601319509897
25   Accuracy:  0.4590009425070688
25   Accuracy:  0.48350612629594725
25   Accuracy:  0.49010367577756836
25   Accuracy:  0.5004712535344015
25   Accuracy:  0.48350612629594725
25   Accuracy:  0.46842601319509897
26   Accuracy:  0.4806786050895382
26   Accuracy:  0.48444863336475025
26   Accuracy:  0.4957587181903864
26   Accuracy:  0.47879359095193214
26   Accur

48   Accuracy:  0.471253534401508
48   Accuracy:  0.4590009425070688
48   Accuracy:  0.4882186616399623
48   Accuracy:  0.4891611687087653
48   Accuracy:  0.49293119698397736
48   Accuracy:  0.4882186616399623
48   Accuracy:  0.4637134778510839
48   Accuracy:  0.47690857681432614
48   Accuracy:  0.48633364750235625
48   Accuracy:  0.5032987747408105
49   Accuracy:  0.4344957587181904
49   Accuracy:  0.48350612629594725
49   Accuracy:  0.4872761545711593
49   Accuracy:  0.4891611687087653
49   Accuracy:  0.4938737040527804
49   Accuracy:  0.4825636192271442
49   Accuracy:  0.47690857681432614
49   Accuracy:  0.5032987747408105
49   Accuracy:  0.46936852026390197
49   Accuracy:  0.4957587181903864
50   Accuracy:  0.49952874646559847
50   Accuracy:  0.4806786050895382
50   Accuracy:  0.4740810556079171
50   Accuracy:  0.46088595664467485
50   Accuracy:  0.4938737040527804
50   Accuracy:  0.46748350612629597
50   Accuracy:  0.46748350612629597
50   Accuracy:  0.5023562676720076
50   Accura

71   Accuracy:  0.46842601319509897
71   Accuracy:  0.4655984919886899
71   Accuracy:  0.47973609802073514
71   Accuracy:  0.47690857681432614
71   Accuracy:  0.5061262959472196
71   Accuracy:  0.49010367577756836
71   Accuracy:  0.5014137606032045
71   Accuracy:  0.4599434495758718
72   Accuracy:  0.46936852026390197
72   Accuracy:  0.47690857681432614
72   Accuracy:  0.47690857681432614
72   Accuracy:  0.46748350612629597
72   Accuracy:  0.46088595664467485
72   Accuracy:  0.4891611687087653
72   Accuracy:  0.4655984919886899
72   Accuracy:  0.5042412818096136
72   Accuracy:  0.529688972667295
72   Accuracy:  0.47879359095193214
73   Accuracy:  0.47973609802073514
73   Accuracy:  0.471253534401508
73   Accuracy:  0.48350612629594725
73   Accuracy:  0.4646559849198869
73   Accuracy:  0.4806786050895382
73   Accuracy:  0.48539114043355325
73   Accuracy:  0.5061262959472196
73   Accuracy:  0.49010367577756836
73   Accuracy:  0.4816211121583412
73   Accuracy:  0.4759660697455231
74   Acc

95   Accuracy:  0.5221489161168709
95   Accuracy:  0.48444863336475025
95   Accuracy:  0.47973609802073514
95   Accuracy:  0.46277097078228085
95   Accuracy:  0.48539114043355325
95   Accuracy:  0.4938737040527804
95   Accuracy:  0.47879359095193214
96   Accuracy:  0.47690857681432614
96   Accuracy:  0.47785108388312914
96   Accuracy:  0.4637134778510839
96   Accuracy:  0.5004712535344015
96   Accuracy:  0.47879359095193214
96   Accuracy:  0.4806786050895382
96   Accuracy:  0.4872761545711593
96   Accuracy:  0.46277097078228085
96   Accuracy:  0.5089538171536286
96   Accuracy:  0.47879359095193214
97   Accuracy:  0.4759660697455231
97   Accuracy:  0.48444863336475025
97   Accuracy:  0.47973609802073514
97   Accuracy:  0.47690857681432614
97   Accuracy:  0.46748350612629597
97   Accuracy:  0.49293119698397736
97   Accuracy:  0.473138548539114
97   Accuracy:  0.49293119698397736
97   Accuracy:  0.49670122525918947
97   Accuracy:  0.47879359095193214
98   Accuracy:  0.46842601319509897
98

18   Accuracy:  0.4654255319148936
19   Accuracy:  0.4844995571302037
19   Accuracy:  0.4720992028343667
19   Accuracy:  0.4641275465013286
19   Accuracy:  0.4428697962798937
19   Accuracy:  0.46058458813108943
19   Accuracy:  0.491585473870682
19   Accuracy:  0.49689991142604073
19   Accuracy:  0.4716312056737589
19   Accuracy:  0.4645390070921986
19   Accuracy:  0.4627659574468085
20   Accuracy:  0.46944198405668736
20   Accuracy:  0.43844109831709477
20   Accuracy:  0.46944198405668736
20   Accuracy:  0.48892825509300264
20   Accuracy:  0.49867139061116034
20   Accuracy:  0.46767050487156775
20   Accuracy:  0.4853852967227635
20   Accuracy:  0.46099290780141844
20   Accuracy:  0.46187943262411346
20   Accuracy:  0.47074468085106386
21   Accuracy:  0.4871567759078831
21   Accuracy:  0.4508414526129318
21   Accuracy:  0.47032772364924713
21   Accuracy:  0.48272807794508416
21   Accuracy:  0.4738706820194863
21   Accuracy:  0.4782993799822852
21   Accuracy:  0.4534986713906112
21   Acc

... (26 outputs left out)

14   Accuracy:  0.42857142857142855
14   Accuracy:  0.39285714285714285
14   Accuracy:  0.5714285714285714
14   Accuracy:  0.5714285714285714
14   Accuracy:  0.5357142857142857
14   Accuracy:  0.39285714285714285
15   Accuracy:  0.4482758620689655
15   Accuracy:  0.4827586206896552
15   Accuracy:  0.4827586206896552
15   Accuracy:  0.4827586206896552
15   Accuracy:  0.42857142857142855
15   Accuracy:  0.4642857142857143
15   Accuracy:  0.5357142857142857
15   Accuracy:  0.6071428571428571
15   Accuracy:  0.5
15   Accuracy:  0.42857142857142855
16   Accuracy:  0.4482758620689655
16   Accuracy:  0.4827586206896552
16   Accuracy:  0.5172413793103449
16   Accuracy:  0.5172413793103449
16   Accuracy:  0.5714285714285714
16   Accuracy:  0.32142857142857145
16   Accuracy:  0.6071428571428571
16   Accuracy:  0.5
16   Accuracy:  0.5357142857142857
16   Accuracy:  0.35714285714285715
17   Accuracy:  0.4482758620689655
17   Accuracy:  0.5862068965517241
17   Accuracy:  0.3448275862068966
17   Acc

39   Accuracy:  0.4827586206896552
39   Accuracy:  0.4827586206896552
39   Accuracy:  0.4827586206896552
39   Accuracy:  0.5517241379310345
39   Accuracy:  0.35714285714285715
39   Accuracy:  0.42857142857142855
39   Accuracy:  0.6071428571428571
39   Accuracy:  0.5
39   Accuracy:  0.42857142857142855
39   Accuracy:  0.5357142857142857
40   Accuracy:  0.4482758620689655
40   Accuracy:  0.4827586206896552
40   Accuracy:  0.4482758620689655
40   Accuracy:  0.4827586206896552
40   Accuracy:  0.35714285714285715
40   Accuracy:  0.6071428571428571
40   Accuracy:  0.5
40   Accuracy:  0.5357142857142857
40   Accuracy:  0.5
40   Accuracy:  0.5
41   Accuracy:  0.4827586206896552
41   Accuracy:  0.41379310344827586
41   Accuracy:  0.4827586206896552
41   Accuracy:  0.3793103448275862
41   Accuracy:  0.4642857142857143
41   Accuracy:  0.5357142857142857
41   Accuracy:  0.5
41   Accuracy:  0.4642857142857143
41   Accuracy:  0.5357142857142857
41   Accuracy:  0.6071428571428571
42   Accuracy:  0.48

64   Accuracy:  0.41379310344827586
64   Accuracy:  0.5172413793103449
64   Accuracy:  0.41379310344827586
64   Accuracy:  0.5172413793103449
64   Accuracy:  0.5714285714285714
64   Accuracy:  0.35714285714285715
64   Accuracy:  0.5357142857142857
64   Accuracy:  0.4642857142857143
64   Accuracy:  0.4642857142857143
64   Accuracy:  0.6071428571428571
65   Accuracy:  0.4482758620689655
65   Accuracy:  0.41379310344827586
65   Accuracy:  0.4482758620689655
65   Accuracy:  0.41379310344827586
65   Accuracy:  0.5357142857142857
65   Accuracy:  0.5714285714285714
65   Accuracy:  0.39285714285714285
65   Accuracy:  0.4642857142857143
65   Accuracy:  0.6785714285714286
65   Accuracy:  0.5
66   Accuracy:  0.3793103448275862
66   Accuracy:  0.5172413793103449
66   Accuracy:  0.5172413793103449
66   Accuracy:  0.5517241379310345
66   Accuracy:  0.39285714285714285
66   Accuracy:  0.5
66   Accuracy:  0.39285714285714285
66   Accuracy:  0.5714285714285714
66   Accuracy:  0.32142857142857145
66   A

89   Accuracy:  0.3793103448275862
89   Accuracy:  0.5357142857142857
89   Accuracy:  0.4642857142857143
89   Accuracy:  0.5714285714285714
89   Accuracy:  0.32142857142857145
89   Accuracy:  0.4642857142857143
89   Accuracy:  0.39285714285714285
90   Accuracy:  0.5517241379310345
90   Accuracy:  0.5862068965517241
90   Accuracy:  0.3103448275862069
90   Accuracy:  0.4827586206896552
90   Accuracy:  0.5
90   Accuracy:  0.5
90   Accuracy:  0.5
90   Accuracy:  0.5
90   Accuracy:  0.4642857142857143
90   Accuracy:  0.4642857142857143
91   Accuracy:  0.5172413793103449
91   Accuracy:  0.6896551724137931
91   Accuracy:  0.3793103448275862
91   Accuracy:  0.5862068965517241
91   Accuracy:  0.42857142857142855
91   Accuracy:  0.4642857142857143
91   Accuracy:  0.5714285714285714
91   Accuracy:  0.42857142857142855
91   Accuracy:  0.2857142857142857
91   Accuracy:  0.5
92   Accuracy:  0.27586206896551724
92   Accuracy:  0.41379310344827586
92   Accuracy:  0.4482758620689655
92   Accuracy:  0.5

In [25]:
print(len(y),"\t Accuracy for dataset_gen: \t", accuracy[0])
print(len(dataset_all_gen_y),"\t Accuracy for dataset_all_gen: \t", accuracy[1])
print(len(dataset_all_original_y),"\t Accuracy for dataset_all_original: \t", accuracy[2])
print(len(dataset_jaccard_gen_y),"\t Accuracy for dataset_jaccard_gen: \t", accuracy[3])
print(len(dataset_jaccard_original_y),"\t Accuracy for dataset_jaccard_original: \t", accuracy[4])
print(len(dataset_myroom_gen_y),"\t Accuracy for dataset_myroom_gen: \t", accuracy[5])
print(len(dataset_myroom_original_y),"\t Accuracy for dataset_myroom_original: \t", accuracy[6])
print(len(dataset_university_gen_y),"\t Accuracy for dataset_university_gen: \t", accuracy[7])
print(len(dataset_university_original_y),"\t Accuracy for dataset_university_original: \t", accuracy[8])

10610 	 Accuracy for dataset_gen: 	 0.4818620012557796
11287 	 Accuracy for dataset_all_gen: 	 0.4802062969020717
677 	 Accuracy for dataset_all_original: 	 0.4891409789648476
10912 	 Accuracy for dataset_jaccard_gen: 	 0.49747291513111297
302 	 Accuracy for dataset_jaccard_original: 	 0.5109045745746633
10701 	 Accuracy for dataset_myroom_gen: 	 0.524800218762553
91 	 Accuracy for dataset_myroom_original: 	 0.5373844140698012
10894 	 Accuracy for dataset_university_gen: 	 0.5486181159278442
284 	 Accuracy for dataset_university_original: 	 0.5554521722872252


In [26]:
print(len(y))
print(len(dataset_all_gen_y))
print(len(dataset_all_original_y))
print(len(dataset_jaccard_gen_y))
print(len(dataset_jaccard_original_y))
print(len(dataset_myroom_gen_y))
print(len(dataset_myroom_original_y))
print(len(dataset_university_gen_y))
print(len(dataset_university_original_y))

10610
11287
677
10912
302
10701
91
10894
284
